In [ ]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# FASE: Paso 2 — Extracción de bloques de ocupación (CORREGIDO)

#    ANTES: first_occ_dt se calculaba sobre la semana entera
#             → Minutes_Since_First_Occupancy se acumulaba entre días
#    AHORA: todas las variables de tiempo se resetean cada día
#
# Variables de tiempo (se resetean por día):
#   - Duration_min              : duración del bloque actual (minutos)
#   - Minutes_Since_DayStart    : minutos desde las 00:00 del día
#   - Minutes_Since_First_Occ   : minutos desde la 1ª ocupación >0 del DÍA
#   - Minutes_Since_CO2_Peak    : minutos desde el pico máx de CO2 del DÍA
# ===================================================================

import pandas as pd
import numpy as np
import os
from openpyxl.styles import PatternFill, Font
import warnings
warnings.filterwarnings('ignore')

INPUT_DIR  = '.'
OUTPUT_DIR = 'occupancy_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']


# ===================================================================
# FUNCIÓN PRINCIPAL — RESET DIARIO DE VARIABLES DE TIEMPO
# ===================================================================
def extract_occupancy_blocks_with_time(df_week, week_name):
    """
    Extrae bloques de ocupación con variables de tiempo reseteadas por día.

    Variables de tiempo (todas reseteadas al inicio de cada día):
      - Duration_min              : duración del bloque (minutos)
      - Minutes_Since_DayStart    : minutos desde 00:00 del día
      - Minutes_Since_First_Occ   : minutos desde primera ocupación >0 del DÍA
      - Minutes_Since_CO2_Peak    : minutos desde el pico máximo de CO2 del DÍA
    """
    occ_row = df_week[df_week['Variable_Measure'] == 'Occupancy']
    if occ_row.empty:
        return []

    occ_row    = occ_row.iloc[0]
    date_cols  = sorted([c for c in df_week.columns if c not in metadata_cols])
    date_cols_dt = [pd.to_datetime(c) for c in date_cols]

    # ── Construir secuencia de ocupación ────────────────────────────
    occupancy_sequence = []
    for col, dt in zip(date_cols, date_cols_dt):
        val = occ_row[col]
        if pd.notna(val):
            try:
                occupancy_sequence.append({'DateTime': dt, 'Occupancy': int(val)})
            except:
                continue

    if not occupancy_sequence:
        return []

    # ── Obtener datos de CO2 del pasillo ────────────────────────────
    co2_row = df_week[
        (df_week['Variable_Measure'] == 'CO2') &
        (df_week['Sensor_Location'].str.contains('Corridor', case=False, na=False))
    ]
    if co2_row.empty:
        # Fallback: cualquier fila de CO2
        co2_row = df_week[df_week['Variable_Measure'] == 'CO2']

    # ── Agrupar secuencia de ocupación por DÍA ──────────────────────
    from collections import defaultdict
    day_sequences = defaultdict(list)
    for entry in occupancy_sequence:
        day = entry['DateTime'].date()
        day_sequences[day].append(entry)

    # ── Datos de sensores (excluir ocupación, puertas, ventanas) ────
    sensor_data = df_week[
        ~df_week['Variable_Measure'].isin(['Occupancy', 'Door_Open', 'Window_Open'])
    ]

    all_rows = []

    for day, day_seq in sorted(day_sequences.items()):

        # ── Calcular first_occ_dt para ESTE DÍA ─────────────────────
        first_occ_dt = None
        for entry in day_seq:
            if entry['Occupancy'] > 0:
                first_occ_dt = entry['DateTime']
                break

        # ── Calcular pico de CO2 para ESTE DÍA ──────────────────────
        co2_peak_dt = None
        if not co2_row.empty:
            co2_vals = []
            for col, dt in zip(date_cols, date_cols_dt):
                if dt.date() != day:
                    continue
                val = co2_row.iloc[0][col]
                if pd.notna(val):
                    try:
                        co2_vals.append((dt, float(val)))
                    except:
                        pass
            if co2_vals:
                co2_peak_dt = max(co2_vals, key=lambda x: x[1])[0]

        # ── Agrupar en bloques por cambio de ocupación ───────────────
        blocks = []
        current_block = [day_seq[0]]
        for i in range(1, len(day_seq)):
            if day_seq[i]['Occupancy'] != day_seq[i-1]['Occupancy']:
                blocks.append(current_block)
                current_block = [day_seq[i]]
            else:
                current_block.append(day_seq[i])
        blocks.append(current_block)

        # ── Procesar cada bloque ─────────────────────────────────────
        for block in blocks:
            block_start = block[0]['DateTime']
            block_end   = block[-1]['DateTime']

            block_indices = [
                i for i, dt in enumerate(date_cols_dt)
                if block_start <= dt <= block_end
            ]
            if not block_indices:
                continue

            # Variables de tiempo — todas relativas al DÍA
            duration_min = round(
                (block_end - block_start).total_seconds() / 60
            )

            day_start_dt = pd.Timestamp(day)
            minutes_since_day_start = round(
                (block_start - day_start_dt).total_seconds() / 60
            )

            if first_occ_dt is not None and block_start >= first_occ_dt:
                minutes_since_first_occ = round(
                    (block_start - first_occ_dt).total_seconds() / 60
                )
            else:
                minutes_since_first_occ = 0  # antes de la primera ocupación del día

            if co2_peak_dt is not None and block_start >= co2_peak_dt:
                minutes_since_co2_peak = round(
                    (block_start - co2_peak_dt).total_seconds() / 60
                )
            elif co2_peak_dt is not None and block_start < co2_peak_dt:
                # Bloque ANTES del pico → negativo (el pico aún no ha llegado)
                minutes_since_co2_peak = -round(
                    (co2_peak_dt - block_start).total_seconds() / 60
                )
            else:
                minutes_since_co2_peak = 0

            row = {
                'Occupancy':                   block[0]['Occupancy'],
                'Date':                        block_start.date(),
                'Day_of_Week':                 block_start.strftime('%A'),
                'Start_Time':                  block_start.time(),
                'End_Time':                    block_end.time(),
                'Duration_min':                duration_min,
                'Minutes_Since_DayStart':      minutes_since_day_start,      # ✅ nuevo
                'Minutes_Since_First_Occ':     minutes_since_first_occ,      # ✅ reseteado por día
                'Minutes_Since_CO2_Peak':      minutes_since_co2_peak,       # ✅ reseteado por día
                'Week':                        week_name,
            }

            # ── Estadísticas de sensores + tendencia ─────────────────
            for _, sensor_row in sensor_data.iterrows():
                values = []
                for idx in block_indices:
                    val = sensor_row[date_cols[idx]]
                    if pd.notna(val):
                        try:
                            values.append(float(val))
                        except:
                            pass

                if values:
                    loc  = sensor_row.get('Sensor_Location', '')
                    var  = sensor_row['Variable_Measure']
                    prefix = f"{loc}_{var}" if loc else var

                    if len(values) > 1:
                        t0 = date_cols_dt[block_indices[0]]
                        t1 = date_cols_dt[block_indices[-1]]
                        span_min = (t1 - t0).total_seconds() / 60
                        trend = (values[-1] - values[0]) / span_min if span_min > 0 else 0
                    else:
                        trend = 0

                    row.update({
                        f'{prefix}_mean':  np.mean(values),
                        f'{prefix}_std':   np.std(values) if len(values) > 1 else 0,
                        f'{prefix}_max':   np.max(values),
                        f'{prefix}_min':   np.min(values),
                        f'{prefix}_trend': round(trend, 4),
                    })

            all_rows.append(row)

    return all_rows


# ===================================================================
# CARGAR ARCHIVOS SEMANALES Y EXTRAER BLOQUES
# ===================================================================
weekly_files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.startswith('1.5_data_') and f.endswith('.xlsx')
])
print(f"Encontrados {len(weekly_files)} archivos semanales")

all_occupancy_rows = []
for fname in weekly_files:
    print(f"   Procesando {fname}...")
    df_week   = pd.read_excel(os.path.join(INPUT_DIR, fname))
    week_name = fname.replace('1.5_data_', '').replace('.xlsx', '')
    rows = extract_occupancy_blocks_with_time(df_week, week_name)
    all_occupancy_rows.extend(rows)
    print(f"      Bloques encontrados: {len(rows)}")

# ===================================================================
# GUARDAR RESULTADOS
# ===================================================================
if all_occupancy_rows:
    df_occ = pd.DataFrame(all_occupancy_rows)

    # Reordenar columnas: metadatos primero, sensores después
    first_cols = [
        'Occupancy', 'Date', 'Day_of_Week', 'Start_Time', 'End_Time',
        'Duration_min',
        'Minutes_Since_DayStart',       # ✅ nuevo
        'Minutes_Since_First_Occ',      # ✅ reseteado por día
        'Minutes_Since_CO2_Peak',       # ✅ reseteado por día
        'Week',
    ]
    other_cols = sorted([c for c in df_occ.columns if c not in first_cols])
    df_occ = df_occ[first_cols + other_cols].sort_values(['Date', 'Start_Time'])

    # ── Verificación del reset ───────────────────────────────────────
    print("\n✅ VERIFICACIÓN DEL RESET DIARIO:")
    print(f"   {'Fecha':<12}  {'Start':<8}  {'Since_DayStart':>14}  "
          f"{'Since_1stOcc':>12}  {'Since_CO2Peak':>13}  {'Ocupacion':>9}")
    print(f"   {'-'*75}")
    prev_day = None
    for _, r in df_occ.head(30).iterrows():
        day = str(r['Date'])
        if day != prev_day:
            print(f"   {'─'*75}  ← nuevo día")
            prev_day = day
        print(f"   {day:<12}  {str(r['Start_Time'])[:5]:<8}  "
              f"{r['Minutes_Since_DayStart']:>14.0f}  "
              f"{r['Minutes_Since_First_Occ']:>12.0f}  "
              f"{r['Minutes_Since_CO2_Peak']:>13.0f}  "
              f"{r['Occupancy']:>9.0f}")

    # ── Guardar Excel ────────────────────────────────────────────────
    file_final = os.path.join(OUTPUT_DIR, '1.5_occupancy_blocks_analysis.xlsx')
    with pd.ExcelWriter(file_final, engine='openpyxl') as writer:
        df_occ.to_excel(writer, sheet_name='Occupancy_Blocks', index=False)
        ws = writer.sheets['Occupancy_Blocks']

        # Cabecera azul
        for cell in ws[1]:
            cell.fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
            cell.font = Font(bold=True, size=11, color='FFFFFF')

        # Color por ocupación
        occ_col = list(df_occ.columns).index('Occupancy') + 1
        for row in range(2, ws.max_row + 1):
            val = ws.cell(row=row, column=occ_col).value
            if val is not None:
                if val == 0:       color = 'F2F2F2'
                elif val <= 20:    color = 'E2EFDA'
                elif val <= 30:    color = 'FCE4D6'
                else:              color = 'F4B4C2'
                ws.cell(row=row, column=occ_col).fill = PatternFill(
                    start_color=color, end_color=color, fill_type='solid')

    print(f"\n   Guardado en: {file_final}")
    print(f"   Total bloques: {len(df_occ)}")
    print(f"   Rango fechas:  {df_occ['Date'].min()} — {df_occ['Date'].max()}")
    print(f"\n   Columnas generadas ({len(df_occ.columns)}):")
    for col in df_occ.columns:
        if 'Minutes' in col or 'Duration' in col:
            marca = " 🕐 TIEMPO (reseteado por día)"
        elif '_trend' in col:
            marca = " 📈 TENDENCIA"
        else:
            marca = ""
        print(f"      - {col}{marca}")

else:
    print("⚠️  No se encontraron bloques de ocupación.")

print("\n✅ PASO 2 COMPLETADO (con reset diario correcto)")

Encontrados 12 archivos semanales
   Procesando 1.5_data_December_2025_week_4_days_22-31.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_February_2026_week_1_days_1-7.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_February_2026_week_2_days_8-14.xlsx...
      Bloques encontrados: 43
   Procesando 1.5_data_February_2026_week_3_days_15-21.xlsx...
      Bloques encontrados: 15
   Procesando 1.5_data_February_2026_week_4_days_22-28.xlsx...
      Bloques encontrados: 17
   Procesando 1.5_data_January_2026_week_1_days_1-7.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_January_2026_week_2_days_8-14.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_January_2026_week_3_days_15-21.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_January_2026_week_4_days_22-31.xlsx...
      Bloques encontrados: 0
   Procesando 1.5_data_March_2026_week_1_days_1-7.xlsx...
      Bloques encontrados: 36
   Procesando 1.5_data_March_2026_week_2_days_8-14.xlsx..